# Все каналы => пул => ранкер => сабмит

Исторические ноутбуки сохранены. Один split и один генератор признаков для обучения, проверки и benchmark. Метки outer-val не попадают в history ранкера.

In [7]:
from pipeline.pool_experiments import PoolExperiment, FEATURES
experiment = PoolExperiment(".", sample_size=5000)
print("features:", len(FEATURES))

history=448040, eval=5000, items=344825
features: 64


## Общий пул

Weighted BM25 3:1:1, отдельные BM25-поля, строгий локальный BM25, эмбеды, lookup, fuzzy, char TF-IDF, гео и фильтры.\
 Геопризнаки доступны каждому кандидату.\
  Выбор top-500 идёт после общего скоринга.

In [2]:
channel_table = experiment.compare()
display(channel_table.round(6))
print("selected pool:", experiment.pool_variant)

Подготовка каналов: 5000 запросов
E5: кодирование 5000 запросов
каналы: 500/5000
каналы: 1000/5000
каналы: 1500/5000
каналы: 2000/5000
каналы: 2500/5000
каналы: 3000/5000
каналы: 3500/5000
каналы: 4000/5000
каналы: 4500/5000
каналы: 5000/5000
selected pool: all_geo_weight_1


Batches: 100%|##########| 157/157 [00:10<00:00, 14.70it/s]


,Recall@50,Recall@500,pool_ceiling,mean_pool_size
variant,,,,
old_score,0.333235,0.648932,0.648932,496.9212
fixed_geo_score,0.477188,0.739201,0.739201,496.9212
weighted_bm25,0.552337,0.790391,0.790391,496.9212
local_bm25,0.559711,0.809984,0.809984,496.9212
plus_fuzzy,0.541128,0.807172,0.807172,497.0364
plus_char,0.532739,0.804395,0.804395,498.2962
all_channels,0.532314,0.804293,0.804293,499.2216
all_geo_weight_0.5,0.750567,0.902925,0.902925,499.2216
all_geo_weight_1,0.815249,0.927362,0.927362,499.2216


## Обучение и одинаковая проверка

Оба ранкера обучаются на внутреннем split истории. Позитивы не добавляются в пул принудительно. Старый контроль использует прежние шесть каналов и признаки.

In [3]:
validation_training = experiment.train_ranker(fit_queries=6000)
print(validation_training)

Подготовка каналов: 6000 запросов
E5: кодирование 6000 запросов
каналы: 500/6000
каналы: 1000/6000
каналы: 1500/6000
каналы: 2000/6000
каналы: 2500/6000
каналы: 3000/6000
каналы: 3500/6000
каналы: 4000/6000
каналы: 4500/6000
каналы: 5000/6000
каналы: 5500/6000
каналы: 6000/6000
0:	learn: 0.4891002	test: 0.4893652	best: 0.4893652 (0)	total: 737ms	remaining: 3m 3s
50:	learn: 0.0623949	test: 0.0729907	best: 0.0729907 (50)	total: 34.3s	remaining: 2m 13s
100:	learn: 0.0514971	test: 0.0683484	best: 0.0683484 (100)	total: 1m 5s	remaining: 1m 36s
150:	learn: 0.0457225	test: 0.0674868	best: 0.0674747 (149)	total: 1m 36s	remaining: 1m 3s
200:	learn: 0.0404084	test: 0.0666639	best: 0.0665585 (187)	total: 2m 6s	remaining: 30.9s
249:	learn: 0.0349167	test: 0.0670661	best: 0.0664487 (221)	total: 2m 36s	remaining: 0us

bestTest = 0.06644873835
bestIteration = 221

Shrink model to first 222 iterations.
0:	learn: 0.5066237	test: 0.5083854	best: 0.5083854 (0)	total: 286ms	remaining: 1m 11s
50:	learn: 0.

Batches: 100%|##########| 188/188 [00:12<00:00, 14.64it/s]


In [4]:
validation_table, warm_cold, context_counts = experiment.evaluate_ranker()
display(validation_table.round(6))
display(warm_cold.round(6))
print(context_counts)

{'warm_contexts': 4364, 'cold_contexts': 636}


,Recall@50,Recall@500,pool_ceiling,mean_pool_size
variant,,,,
old_score,0.333235,0.648932,0.648932,496.9212
fixed_geo_score,0.477188,0.739201,0.739201,496.9212
weighted_bm25,0.552337,0.790391,0.790391,496.9212
local_bm25,0.559711,0.809984,0.809984,496.9212
plus_fuzzy,0.541128,0.807172,0.807172,497.0364
plus_char,0.532739,0.804395,0.804395,498.2962
all_channels,0.532314,0.804293,0.804293,499.2216
all_geo_weight_0.5,0.750567,0.902925,0.902925,499.2216
all_geo_weight_1,0.815249,0.927362,0.927362,499.2216


,warm_recall50,cold_recall50
variant,,
old_score,0.321458,0.414046
fixed_geo_score,0.468478,0.536950
weighted_bm25,0.548164,0.580975
local_bm25,0.556841,0.579403
plus_fuzzy,0.537039,0.569182
plus_char,0.526130,0.578092
all_channels,0.525184,0.581237
all_geo_weight_0.5,0.752445,0.737683
all_geo_weight_1,0.821207,0.774371


## Проверка на бенче

Берём outer-val контексты с известным релевантным item в benchmark.\
 Recall условный: в знаменателе только такие item. Это proxy, а не скор платформы. 70% контекстов выбирают политику, 30% проверяют её; отдельно warm/cold. Итоговая смесь приближает долю новых текстов benchmark: 63%.

In [8]:
benchmark_table = experiment.evaluate_benchmark_proxy()
display(benchmark_table.round(6))
print("submission policy:", experiment.final_policy)

benchmark proxy: 2679 контекстов, только релевантные item benchmark
Подготовка каналов: 2679 запросов
каналы: 500/2679
каналы: 1000/2679
каналы: 1500/2679
каналы: 2000/2679
каналы: 2500/2679
Политика для submission: ranker_heuristic_geo_rrf_2
submission policy: ranker_heuristic_geo_rrf_2


,subset,variant,contexts,cold_contexts,Recall@50,Recall@500,warm_recall50,cold_recall50,benchmark_mix_recall50
0,all,updated_catboost,2679,346,0.822954,0.968583,0.833131,0.754335,0.783490
1,all,weighted_geo50,2679,346,0.832564,0.832564,0.837737,0.797688,0.812506
2,all,pool_heuristic,2679,346,0.867107,0.968583,0.876545,0.803468,0.830506
3,all,old_catboost,2679,346,0.658815,0.762895,0.658793,0.658960,0.658898
4,all,ranker_geo_rrf_0.25,2679,346,0.839845,0.968210,0.849098,0.777457,0.803964
5,all,ranker_heuristic_rrf_0.25,2679,346,0.838819,0.968583,0.847919,0.777457,0.803528
6,all,ranker_heuristic_geo_rrf_0.25,2679,346,0.853840,0.968583,0.860454,0.809249,0.828194
7,all,ranker_geo_rrf_0.5,2679,346,0.847232,0.968210,0.852865,0.809249,0.825387
8,all,ranker_heuristic_rrf_0.5,2679,346,0.846004,0.968583,0.855313,0.783237,0.809905
9,all,ranker_heuristic_geo_rrf_0.5,2679,346,0.858164,0.968583,0.864990,0.812139,0.831694


## Финальная модель и CSV

Повторяем внутренний split всего train. Статистики submission строятся на всём train, item-корпус — benchmark. Проверяем ровно 50 уникальных допустимых item на каждый query_id. v1 не перезаписывается.

In [9]:
production_training = experiment.train_ranker(fit_queries=8000, production=True)
print(production_training)

production: готово 0, осталось 8000 контекстов
Подготовка каналов: 8000 запросов
каналы: 500/8000
каналы: 1000/8000
каналы: 1500/8000
каналы: 2000/8000
каналы: 2500/8000
каналы: 3000/8000
каналы: 3500/8000
каналы: 4000/8000
каналы: 4500/8000
каналы: 5000/8000
каналы: 5500/8000
каналы: 6000/8000
каналы: 6500/8000
каналы: 7000/8000
каналы: 7500/8000
каналы: 8000/8000
0:	learn: 0.4983275	test: 0.4949857	best: 0.4949857 (0)	total: 946ms	remaining: 3m 55s
50:	learn: 0.0509785	test: 0.0501117	best: 0.0501117 (50)	total: 44.5s	remaining: 2m 53s
100:	learn: 0.0427632	test: 0.0466487	best: 0.0466018 (95)	total: 1m 25s	remaining: 2m 6s
150:	learn: 0.0377321	test: 0.0456698	best: 0.0455156 (145)	total: 2m 6s	remaining: 1m 22s
200:	learn: 0.0334873	test: 0.0453128	best: 0.0452207 (188)	total: 2m 47s	remaining: 40.7s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.04512379639
bestIteration = 204

Shrink model to first 205 iterations.
0:	learn: 0.4496782	test: 0.4484465	best: 0.4

In [10]:
submission_metadata = experiment.make_submission()
print(submission_metadata)

Подготовка каналов: 2452 запросов
E5: кодирование 2452 запросов
каналы: 500/2452
каналы: 1000/2452
каналы: 1500/2452
каналы: 2000/2452
{'file': 'submissions/v2_all_channels.csv', 'queries': 2452, 'items_per_query': 50, 'pool_variant': 'all_geo_weight_1', 'final_policy': 'ranker_heuristic_geo_rrf_2', 'features': ['bm25_norm', 'geo_distance_km', 'geo_bonus', 'geo_exact', 'vid_match', 'tip_match', 'log_popularity', 'fuzzy_score', 'char_score', 'dense_score', 'title_coverage', 'item_price', 'item_rating', 'log_reviews', 'geo_exact_match', 'vid_uslugi_match', 'tip_uslugi_match', 'item_rating_reviews_count', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_popularity', 'text_overlap', 'bm25_rank', 'bm25_present', 'bm25_score', 'dense_rank', 'dense_present', 'lookup_rank', 'lookup_present', 'lookup_score', 'geo_rank', 'geo_present', 'geo_score', 'filter_rank', 'filter_present', 'filter_score', 'geo_exact_rank', 'geo_exact_present', 'geo_exact_score', 'weighted_rank', 'weighted_prese

Batches: 100%|##########| 77/77 [00:06<00:00, 12.34it/s]


## Результат

Финальный CSV: `submissions/v<n>.csv`, 2452 запроса по 50 уникальных item. Модель: 205 деревьев, 64 признака. Смесь ранкер/эвристика/гео = 1/2/0.5 выбрана на benchmark-proxy. Метрики ниже получены на validation-модели, production-модель обучена на внутреннем split всего train. Benchmark-proxy учитывает только известные релевантные item этого каталога; это не скор платформы.


In [11]:
import json
from pathlib import Path
import pandas as pd
from catboost import CatBoostRanker
from pipeline.pool_experiments import FEATURES

old_scores = pd.read_csv("data/pool_eval/final_comparison.csv", index_col=0)
final_scores = pd.read_csv("data/pool_eval/submission_validation.csv").iloc[0]
proxy = pd.read_csv("data/pool_eval/benchmark_proxy.csv")
control = proxy.loc[proxy["subset"] == "policy_holdout"].set_index("variant")
summary = pd.DataFrame([
    {"comparison": "Recall@50: 5000 contexts, final policy", "old": old_scores.loc["old_control_catboost", "Recall@50"], "new": final_scores["Recall@50"]},
    {"comparison": "Recall@50: benchmark proxy, 804 contexts", "old": control.loc["old_catboost", "Recall@50"], "new": control.loc["ranker_heuristic_geo_rrf_2", "Recall@50"]},
    {"comparison": "Pool ceiling: 5000 contexts", "old": old_scores.loc["old_control_catboost", "pool_ceiling"], "new": old_scores.loc["updated_catboost", "pool_ceiling"]},
])
display(summary.round(6))
model = CatBoostRanker()
model.load_model("data/pool_eval/production_ranker.cbm")
importance = pd.DataFrame({"feature": FEATURES, "importance": model.get_feature_importance(type="PredictionValuesChange")}).sort_values("importance", ascending=False)
importance.to_csv("data/pool_eval/production_feature_importance.csv", index=False)
display(importance.head(12).round(4))
print(json.loads(Path("data/pool_eval/run_summary.json").read_text()))


{'submission': 'submissions/v2_all_channels.csv', 'queries': 2452, 'unique_items_per_query': 50, 'all_ids_in_benchmark': True, 'v1_unchanged': True, 'model_trees': 205, 'features': 64, 'mean_overlap_with_v1': 20.71941272430669, 'final_policy': 'ranker_heuristic_geo_rrf_2', 'sha256': 'ae1d1cfeaea73a50a6272632d92bc4a193f679f1874a810b467a334644533cfd'}


,comparison,old,new
0,"Recall@50: 5000 contexts, final policy",0.512413,0.845070
1,"Recall@50: benchmark proxy, 804 contexts",0.649793,0.856965
2,Pool ceiling: 5000 contexts,0.533668,0.927362


,feature,importance
1,geo_distance_km,19.7275
2,geo_bonus,12.6667
15,vid_uslugi_match,9.2665
42,local_rank,7.2051
0,bm25_norm,6.7671
6,log_popularity,3.3121
8,char_score,2.9744
20,item_popularity,2.9173
51,title_score,2.5247
43,local_present,2.1125
